# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset from Frontiers using the `mlcroissant` library.

### Dataset Source
The dataset schema is provided via a Croissant schema URL.

- **Title:** Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review
- **Identifier:** 10.71728/senscience.cbsv-djdb
- **License:** [Open Data Commons Attribution](https://opendatacommons.org/licenses/by/1-0/)


In [ ]:
# Install mlcroissant if not present
!pip install mlcroissant

## 1. Data Loading
Load the dataset's metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as an object
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their fields, and their IDs (`@id`).

We will enumerate the record sets and extract their IDs for later use.

In [ ]:
# Access record sets: Each has a unique @id
record_sets = []
for record_set in dataset.record_sets:
    print(f"RecordSet @id: {record_set['@id']}")
    print(f"  Name: {record_set.get('name', 'N/A')}")
    print(f"  Description: {record_set.get('description', 'N/A')}")
    print(f"  Fields:")
    for field in record_set.get('fields', []):
        print(f"    Field @id: {field['@id']} (name: {field.get('name', '')})")
    record_sets.append(record_set['@id'])

# Show all extracted record set @id's
print("Available RecordSet @id's:")
print(record_sets)

## 3. Data Extraction
Load each record set into a DataFrame for analysis. Reference by `@id`.

In [ ]:
dataframes = {}

# Iterate over record set @id's and extract their records
for record_set_id in record_sets:
    # The generator yields dictionaries keyed by field @id
    records = list(dataset.records(record_set=record_set_id))
    # Convert to dataframe: columns are field @id's
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"DataFrame for {record_set_id} has columns:")
    print(df.columns.tolist())
    print("\nSample records:")
    print(df.head())


## 4. Exploratory Data Analysis (EDA)
Apply common processing steps: filtering, normalization, and grouping.

Here, we will:
- Select a numeric field (`@id`).
- Filter records above a threshold.
- Normalize the numeric field.
- Group by a categorical field if available.


In [ ]:
# Example: Use the first record set and its fields
if record_sets:
    record_set_id = record_sets[0]
    df = dataframes[record_set_id]
    columns = df.columns.tolist()
    print(f"Fields in {record_set_id}: {columns}")

    # Try to select a numeric field (e.g., age at diagnosis)
    # (Assuming the schema uses something like 'age_at_diagnosis' or similar)
    numeric_field = None
    for col in columns:
        if 'age' in col.lower() or 'height' in col.lower():
            numeric_field = col
            break

    # Use a threshold filter
    threshold = 20
    if numeric_field and pd.api.types.is_numeric_dtype(df[numeric_field]):
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize the field
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
            filtered_df[numeric_field].std()
        )
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to group by a field containing 'phenotype' or 'variant' or other categorical
        group_field = None
        for col in columns:
            if ('phenotype' in col.lower()) or ('variant' in col.lower()) or ('group' in col.lower()):
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"Grouped mean {numeric_field} by {group_field}:")
            print(grouped_df.head())

## 5. Visualization
Visualize data distributions and relationships using matplotlib or seaborn.

In [ ]:
# Example: Histogram of numeric field
import matplotlib.pyplot as plt
import seaborn as sns

if record_sets:
    record_set_id = record_sets[0]
    df = dataframes[record_set_id]
    columns = df.columns.tolist()

    # Try to select a numeric field
    numeric_field = None
    for col in columns:
        if 'age' in col.lower() or 'height' in col.lower():
            numeric_field = col
            break

    if numeric_field:
        plt.figure(figsize=(6, 4))
        sns.histplot(df[numeric_field].dropna(), bins=10, kde=True)
        plt.title(f'Distribution of {numeric_field}')
        plt.xlabel(numeric_field)
        plt.ylabel('Count')
        plt.show()

    # Visualize categorical group if available
    group_field = None
    for col in columns:
        if ('phenotype' in col.lower()) or ('variant' in col.lower()) or ('group' in col.lower()):
            group_field = col
            break

    if numeric_field and group_field:
        plt.figure(figsize=(6, 4))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f'{numeric_field} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
This notebook demonstrated loading, overview, extraction, EDA, and visualization for the FAIR² clinical dataset using `mlcroissant`.

- Accessed metadata and record sets using their `@id`s.
- Loaded multiple record sets and fields for analysis.
- Performed basic filtering and normalization.
- Visualized key clinical numeric distributions.

Due to small sample size (N=3), interpret results with caution. The dataset is intended for academic illustration and genotype–phenotype heterogeneity analysis.